# Entrega 3 — Chunking das manifestações longas

As cinco manifestações mais longas são divididas com `RecursiveCharacterTextSplitter`, conforme solicitado. Duas configurações são comparadas: uma com chunks menores e overlap moderado e outra com chunks maiores e overlap maior. O objetivo é observar coesão, quantidade de pedaços e comportamento no espaço vetorial.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path.cwd()))
from ouvidoria import carregar, dividir, representar, grafico_2d, MODELOS

df=carregar()
longos=df.assign(n=df.texto.str.len()).nlargest(5,"n")
display(longos[["id","n"]])


In [ ]:
configuracoes=[("A",260,40),("B",420,90)]
registros=[]
for _,row in longos.iterrows():
    for nome,tamanho,overlap in configuracoes:
        partes=dividir(row.texto,tamanho,overlap,"Recursivo")
        for i,p in enumerate(partes,1):
            registros.append({"id":row.id,"config":nome,"chunk":i,"caracteres":len(p),"texto":p})
chunks=pd.DataFrame(registros)
display(chunks.groupby("config").agg(chunks=("chunk","count"),media_caracteres=("caracteres","mean")))
chunks.to_csv("resultados/chunks_comparacao.csv",index=False)


In [ ]:
modelo=list(MODELOS)[0]
for config in chunks.config.unique():
    subset=chunks[chunks.config==config]
    emb=representar(tuple(subset.texto),modelo)
    fig=grafico_2d(emb,subset.id.tolist(),"PCA",subset.id.tolist())
    fig.savefig(f"resultados/chunks_{config}.png",dpi=160)
    plt.show()


## Discussão

Overlap maior tende a preservar informação de fronteira: uma frase que começa no fim de um chunk pode reaparecer no seguinte. O custo é gerar mais conteúdo repetido e mais vetores. Chunks maiores carregam mais contexto, mas podem misturar assuntos e reduzir a precisão da recuperação. Nesta proposta, a configuração B é a candidata a preservar mais contexto; a decisão final deve considerar a tabela de tamanhos e a projeção produzida no ambiente de execução.